In [97]:
# Create a multi-band PFT fractions GeoTIFF (0.1°; EPSG:4326), one band per PFT.
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import Affine
from rasterio.crs import CRS
from pathlib import Path

In [98]:
#OUT_DIR = Path("lulc_outputs_final")
#wide_csv = OUT_DIR / r"D:/Reclassified/PFT_fractions_0.05deg_wide.csv"
#df = pd.read_csv(wide_csv)
#PFTS = ["pft1_NLE","pft2_NLD","pft3_BLE","pft45_BCD_BDD","pft67_C3C_C4C",
 #       "pft89_C3G_C4G","pft10_Sedge","pft11_SBE","pft12_SBD","Urban","Lake","Ocean","Bare"]
#lon_vals = np.sort(df["lon0"].unique())
#lat_vals = np.sort(df["lat0"].unique())[::-1]  # descending
#width = lon_vals.size; height = lat_vals.size; dxy = 0.1
#min_lon = float(lon_vals.min()); max_lat = float(lat_vals.max() + dxy)
#transform = Affine(dxy, 0.0, min_lon, 0.0, -dxy, max_lat)

In [99]:
#stack = np.zeros((len(PFTS), height, width), dtype=np.float32)
#lon_to_ix = {v:i for i,v in enumerate(lon_vals)}
#lat_to_iy = {v:i for i,v in enumerate(lat_vals)}
#for row in df.itertuples(index=False):
 #   x = lon_to_ix[row.lon0]; y = lat_to_iy[row.lat0]
  #  for b, col in enumerate(PFTS):
   #     stack[b,y,x] = 0.0 if pd.isna(getattr(row, col)) else float(getattr(row, col))
#tif_path = OUT_DIR /r"D:/Reclassified/PFT_fractions_0.05deg.tif"
#profile = {"driver":"GTiff","height":height,"width":width,"count":len(PFTS),
 #          "dtype":"float32","crs":CRS.from_epsg(4326),"transform":transform,"compress":"LZW"}
#with rasterio.open(tif_path, "w", **profile) as dst:
 #   for b,col in enumerate(PFTS, start=1):
  #      dst.write(stack[b-1,:,:], b)
   #     dst.set_band_description(b, col)
#print("Wrote:", tif_path)

In [100]:
# ========== USER SETTINGS ==========
OUT_DIR = Path(r"D:/Reclassified/PFTs netcdf/")
CSV_PATH = OUT_DIR / r"D:/Reclassified/PFTs csv/2019/PFT_fractions_0.05deg_wide.csv"   # <-- your wide table path

# choose outputs
WRITE_TIFF   = True
WRITE_NETCDF = True

# filenames
TIFF_PATH = OUT_DIR / "PFT_fractions_0.05deg_2019.tif"
NC_PATH   = OUT_DIR / "PFT_fractions_0.05deg_2019.nc"

# treat input lon0/lat0 as cell centers (use half-cell offset in TIFF transform)
CELL_CENTERS = True

In [101]:
# PFT band names (in CSV)
PFTS = [
    "pft1_NLE","pft2_NLD","pft3_BLE","pft45_BCD_BDD","pft67_C3C_C4C",
    "pft89_C3G_C4G","pft10_Sedge","pft11_SBE","pft12_SBD","Urban","Lake","Ocean","Bare"
]
# ===================================

OUT_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(CSV_PATH)

In [102]:
# --- Build grid ---
lon_vals = np.sort(df["lon0"].unique())
lat_vals_asc = np.sort(df["lat0"].unique())       # ascending
lat_vals = lat_vals_asc[::-1]                     # descending for row-major image write

In [103]:
# Detect grid spacing (dxy) from unique coords
def _step(vals):
    diffs = np.diff(np.sort(vals))
    # avoid zeros (duplicates); be robust to tiny noise
    diffs = diffs[np.abs(diffs) > 1e-9]
    return float(np.median(diffs)) if diffs.size else np.nan

d_lon = _step(lon_vals)
d_lat = _step(lat_vals_asc)

In [104]:
# sanity: prefer square cells; if not, we still proceed but warn
if not np.isfinite(d_lon) or not np.isfinite(d_lat):
    raise ValueError("Could not infer grid spacing from lon0/lat0.")
if not np.isclose(d_lon, d_lat, rtol=1e-4, atol=1e-6):
    print(f"Warning: non-square grid detected (d_lon={d_lon}, d_lat={d_lat}). Proceeding.")
dxy = d_lon

width  = lon_vals.size
height = lat_vals.size

lon_to_ix = {v: i for i, v in enumerate(lon_vals)}
lat_to_iy = {v: i for i, v in enumerate(lat_vals)}

In [105]:
# --- Fill stack [pft, lat, lon] ---
stack = np.zeros((len(PFTS), height, width), dtype=np.float32)
for row in df.itertuples(index=False):
    x = lon_to_ix[getattr(row, "lon0")]
    y = lat_to_iy[getattr(row, "lat0")]
    for b, col in enumerate(PFTS):
        val = getattr(row, col)
        stack[b, y, x] = 0.0 if pd.isna(val) else float(val)

In [106]:
# --- Quick integrity check: per-pixel sums ---
pixel_sums = stack.sum(axis=0)  # [lat, lon]
dev = np.abs(pixel_sums - 1.0)
max_dev = float(np.nanmax(dev))
over = int(np.sum(pixel_sums > 1.0001))
under = int(np.sum(pixel_sums < 0.9999))
print(f"Pixel-sum deviation |sum-1| max: {max_dev:.6f} (>{1e-4} for {over} over, {under} under)")

Pixel-sum deviation |sum-1| max: 1.000000 (>0.0001 for 0 over, 771 under)


In [107]:
# =========================
# Write GeoTIFF (multi-band)
# =========================
if WRITE_TIFF:
    import rasterio
    from rasterio.crs import CRS
    from rasterio.transform import from_origin

    # Top-left corner of top-left pixel
    min_lon = float(lon_vals.min())
    max_lat = float(lat_vals.max())  # remember lat_vals is descending

    if CELL_CENTERS:
        x0 = min_lon - dxy/2.0
        y0 = max_lat + dxy/2.0
    else:
        x0 = min_lon
        y0 = max_lat

    transform = from_origin(x0, y0, dxy, dxy)

    profile = {
        "driver": "GTiff",
        "height": height,
        "width": width,
        "count": len(PFTS),
        "dtype": "float32",
        "crs": CRS.from_epsg(4326),
        "transform": transform,
        "compress": "LZW",
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256,
    }

    with rasterio.open(TIFF_PATH, "w", **profile) as dst:
        for b, col in enumerate(PFTS, start=1):
            dst.write(stack[b-1, :, :], b)
            dst.set_band_description(b, col)

    print("Wrote GeoTIFF:", TIFF_PATH)

Wrote GeoTIFF: D:\Reclassified\PFTs netcdf\PFT_fractions_0.05deg_2019.tif


In [108]:
# =========================
# Write NetCDF (CF-friendly)
# =========================
#if WRITE_NETCDF:
 #   import xarray as xr

    # Coordinates use centers exactly as provided
  #  da = xr.DataArray(
   #     stack,
    #    dims=("pft", "lat", "lon"),
     #   coords={"pft": PFTS, "lat": lat_vals, "lon": lon_vals},
      #  name="pft_fraction",
    #)

    #da.attrs.update({
     #   "long_name": "Plant Functional Type fraction",
      #  "units": "1",
       # "comment": "Fractions per pixel for each PFT; rows(lat) are descending to match GeoTIFF write order.",
    #})

    #ds = da.to_dataset()
    #ds.attrs.update({
     #   "title": "PFT Fractions",
      #  "Conventions": "CF-1.8",
       # "grid_mapping": "latitude_longitude",
        #"history": "Created from wide CSV of PFT fractions",
        #"spatial_resolution": f"{dxy} degree",
        #"crs": "EPSG:4326",
    #})

    #encoding = {
     #   "pft_fraction": {
      #      "zlib": True,
       #     "complevel": 4,
        #    "dtype": "float32",
         #   "chunksizes": (len(PFTS), max(1, height//4), max(1, width//4)),
          #  "_FillValue": np.float32(np.nan),
        #}
    #}

    #ds.to_netcdf(NC_PATH, mode="w", format="NETCDF4", encoding=encoding)
    #print("Wrote NetCDF:", NC_PATH)

#print("Done.")

In [109]:
# =========================
# Write NetCDF (CF-friendly + GIS-friendly)
# - Single 3D var with pft dimension (and robust labels)
# - PLUS separate variables per PFT for tools that expect "bands"
# =========================
if WRITE_NETCDF:
    from netCDF4 import Dataset
    import re

    def _sanitize(name: str) -> str:
        # valid CF variable name (letters, digits, _), no spaces or punctuation
        s = re.sub(r'[^0-9A-Za-z_]+', '_', name)
        if re.match(r'^[0-9]', s):
            s = f'v_{s}'
        return s

    NC_PATH.parent.mkdir(parents=True, exist_ok=True)
    with Dataset(NC_PATH, "w", format="NETCDF4") as nc:
        # Dimensions
        nc.createDimension("pft", len(PFTS))
        nc.createDimension("lat", height)
        nc.createDimension("lon", width)

        # For broad compatibility: fixed-length char strings for pft names
        name_strlen = max(len(n) for n in PFTS)
        nc.createDimension("name_strlen", name_strlen)

        # Coordinates
        lat = nc.createVariable("lat", "f4", ("lat",))
        lon = nc.createVariable("lon", "f4", ("lon",))
        lat[:] = lat_vals
        lon[:] = lon_vals
        lat.standard_name = "latitude";  lat.units = "degrees_north"
        lon.standard_name = "longitude"; lon.units = "degrees_east"

        # Numeric PFT ids (1..N) with CF flags + separate char labels
        pft_id = nc.createVariable("pft", "i4", ("pft",))
        pft_id[:] = np.arange(1, len(PFTS) + 1, dtype=np.int32)
        pft_id.long_name = "PFT identifier"
        pft_id.flag_values = np.arange(1, len(PFTS) + 1, dtype=np.int32)
        # CF recommends space-separated meanings
        pft_id.flag_meanings = " ".join(_sanitize(n) for n in PFTS)

        pft_name = nc.createVariable("pft_name", "S1", ("pft", "name_strlen"))
        # write as char array
        for i, name in enumerate(PFTS):
            pft_name[i, :len(name)] = np.array(list(name), dtype="S1")
        pft_name.long_name = "PFT name (char array)"

        # Main 3D variable
        var = nc.createVariable(
            "pft_fraction", "f4", ("pft", "lat", "lon"),
            zlib=True, complevel=4, fill_value=np.float32(np.nan),
            chunksizes=(len(PFTS), max(1, height//4), max(1, width//4))
        )
        var[:] = stack
        var.long_name = "Plant Functional Type fraction"
        var.units = "1"
        var.coordinates = "lat lon pft"  # helps some readers

        # Extra: per-PFT 2D variables (one “band” per PFT)
        for i, name in enumerate(PFTS):
            vname = _sanitize(name)
            v = nc.createVariable(
                vname, "f4", ("lat", "lon"),
                zlib=True, complevel=4, fill_value=np.float32(np.nan),
                chunksizes=(max(1, height//4), max(1, width//4))
            )
            v[:] = stack[i, :, :]
            v.long_name = f"PFT fraction: {name}"
            v.units = "1"

        # Global attrs
        nc.title = "PFT Fractions"
        nc.Conventions = "CF-1.8"
        nc.history = "Created from wide CSV of PFT fractions; includes both pft-dimensioned and per-PFT variables."
        nc.spatial_resolution = f"{dxy} degree"
        nc.crs = "EPSG:4326"

    print("Wrote NetCDF (with pft flags + per-PFT bands):", NC_PATH)

Wrote NetCDF (with pft flags + per-PFT bands): D:\Reclassified\PFTs netcdf\PFT_fractions_0.05deg_2019.nc
